# Practical 06 — Domain-Specific RAG Question Answering
### **Project: Aster Robotics — Internal Operations Knowledge Assistant**

Saumyajeet Singh Bhati - 202401110065

**Aim:** Build a Retrieval-Augmented Generation application from a fictional company's policy documents and compare its answers against a pretrained LLM without retrieval.

**Models:** `all-MiniLM-L6-v2` for embeddings and `google/flan-t5-base` for answer generation. **Vector DB:** ChromaDB with cosine distance. **Domain:** fictional Aster Robotics operations handbook; facts are intentionally invented to make retrieval benefits measurable.

> Run cells top to bottom in Google Colab. CPU works; GPU is faster. Internet is required to download pretrained models. No API key required.

## 1. Install dependencies

In [1]:
!pip -q install "chromadb>=0.5" "sentence-transformers>=3.0" "transformers>=4.44,<5" "accelerate>=0.33" "pypdf>=4" pandas

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-cloud-bigquery 3.47.0 requires python-dateutil<3.0.0,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
holidays 0.108 requires python-dateutil<3,>=2.9.0.post0, but you have python-dateutil 4.0.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
betterproto 2.0.0b7 requires python-dateutil<3.0,>=2.8, but you have python-dateutil 4.0.0 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompat

## 2. Imports and configuration

In [2]:
import os, re, glob, uuid, time
from pathlib import Path
import pandas as pd
import numpy as np
import torch
import chromadb
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

DOCS_DIR = Path('/content/aster_docs')
DB_DIR = '/content/aster_chroma'
EMBED_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'
GEN_MODEL = 'google/flan-t5-base'
CHUNK_SIZE = 480       # characters
CHUNK_OVERLAP = 90    # characters
TOP_K = 3
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Running on:', DEVICE)

Running on: cuda


## 3. Build a domain-specific document collection
The five sample text files below represent a **fictional** company's internal knowledge base. You can also upload `.txt` or `.pdf` files into `/content/aster_docs` and rerun the loading/indexing cells.

In [3]:
SAMPLE_DOCS = {
'hr_policy.txt': """Aster Robotics | Human Resources Policy | 2026
Employees receive 18 days of paid annual leave each calendar year. Leave requests of three or more consecutive working days must be submitted at least 10 calendar days before the start date.
The standard probation period for new employees is 90 days. Employees may work remotely up to two days per week with manager approval.
All new hires must complete workplace safety orientation within their first 14 days.""",
'it_security.txt': """Aster Robotics | Information Security Standard | 2026
Employees must enable multi-factor authentication on every company account. Company passwords must contain at least 14 characters.
Lost laptops must be reported to the IT response desk within 30 minutes of discovery. Access reviews are conducted every 60 days.
Confidential engineering files may be stored only in the approved VaultDrive workspace. Personal cloud drives are not permitted.""",
'engineering.txt': """Aster Robotics | Engineering Workflow | 2026
Every production robot software release requires two independent code reviewers and a passing automated safety test suite.
The company uses a 21-day sprint cycle. Critical production defects must be acknowledged within 20 minutes of alert creation.
The designated release approval role is the Systems Reliability Lead. Deployment rollback exercises take place on the first Friday of each quarter.""",
'facilities.txt': """Aster Robotics | Facilities and Equipment | 2026
Prototype laboratory access is available from 07:00 to 21:00 on weekdays. Visitors must be escorted by an employee at all times.
Protective eyewear is mandatory in the fabrication zone. Equipment incidents must be recorded in the SafetyLog portal within four hours.
The emergency assembly location is the east parking courtyard beside Gate C.""",
'learning.txt': """Aster Robotics | Learning and Development | 2026
Each employee receives a yearly training allowance of 1200 USD. Courses costing more than 300 USD require approval from the employee's team lead.
New engineers must complete the Robot Safety Fundamentals module within 30 days of joining.
Employees who complete an approved certification may claim reimbursement within 45 days after receiving their certificate."""
}
DOCS_DIR.mkdir(parents=True, exist_ok=True)
for filename, content in SAMPLE_DOCS.items():
    path = DOCS_DIR / filename
    if not path.exists(): path.write_text(content, encoding='utf-8')
print('Available documents:', [p.name for p in DOCS_DIR.iterdir()])

Available documents: ['facilities.txt', 'learning.txt', 'engineering.txt', 'it_security.txt', 'hr_policy.txt']


## 4. Load and clean TXT/PDF documents

In [4]:
from pypdf import PdfReader

def clean_text(text):
    text = text.replace('\x00', ' ')
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def load_documents(folder):
    records = []
    for path in sorted(folder.iterdir()):
        if path.suffix.lower() == '.txt':
            raw = path.read_text(encoding='utf-8', errors='replace')
        elif path.suffix.lower() == '.pdf':
            raw = '\n'.join(page.extract_text() or '' for page in PdfReader(str(path)).pages)
        else:
            continue
        cleaned = clean_text(raw)
        if cleaned: records.append({'source': path.name, 'text': cleaned})
    return records

documents = load_documents(DOCS_DIR)
assert documents, 'No documents found!'
pd.DataFrame([{'file': d['source'], 'characters': len(d['text'])} for d in documents])

,file,characters
0,engineering.txt,441
1,facilities.txt,391
2,hr_policy.txt,457
3,it_security.txt,443
4,learning.txt,409


## 5. Chunking with overlap
Character-based overlapping chunks preserve context across boundaries. **480 characters** balances retrieval specificity and context length; **90-character overlap** reduces information loss at chunk boundaries. For larger corpora, token-aware or semantic chunking may be preferable.

In [5]:
def chunk_text(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    if not (0 <= overlap < chunk_size): raise ValueError('overlap must be smaller than chunk size')
    chunks, start = [], 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        if end < len(text):
            boundary = text.rfind('. ', start + chunk_size // 2, end)
            if boundary != -1: end = boundary + 1
        part = text[start:end].strip()
        if part: chunks.append(part)
        if end == len(text): break
        start = max(start + 1, end - overlap)
    return chunks

chunk_records = []
for doc in documents:
    for i, piece in enumerate(chunk_text(doc['text'])):
        chunk_records.append({'id': f"{doc['source']}::{i}", 'source': doc['source'], 'chunk': piece})
print('Documents:', len(documents), '| Chunks:', len(chunk_records))
pd.DataFrame(chunk_records).head()

Documents: 5 | Chunks: 5


,id,source,chunk
0,engineering.txt::0,engineering.txt,Aster Robotics | Engineering Workflow | 2026 E...
1,facilities.txt::0,facilities.txt,Aster Robotics | Facilities and Equipment | 20...
2,hr_policy.txt::0,hr_policy.txt,Aster Robotics | Human Resources Policy | 2026...
3,it_security.txt::0,it_security.txt,Aster Robotics | Information Security Standard...
4,learning.txt::0,learning.txt,Aster Robotics | Learning and Development | 20...


## 6. Generate dense embeddings and create ChromaDB vector index

In [6]:
embedder = SentenceTransformer(EMBED_MODEL, device=DEVICE)
texts = [r['chunk'] for r in chunk_records]
embeddings = embedder.encode(texts, normalize_embeddings=True, show_progress_bar=True).tolist()
print('Embedding dimensions:', len(embeddings[0]))
client = chromadb.PersistentClient(path=DB_DIR)
collection_name = 'aster_robotics_knowledge'
try: client.delete_collection(collection_name)
except Exception: pass
collection = client.create_collection(name=collection_name, metadata={'hnsw:space': 'cosine'})
collection.add(
    ids=[r['id'] for r in chunk_records],
    documents=texts,
    metadatas=[{'source': r['source']} for r in chunk_records],
    embeddings=embeddings
)
print('Indexed chunks:', collection.count())

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (4 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding dimensions: 384
Indexed chunks: 5


## 7. Semantic retrieval with similarity scores and source attribution

In [7]:
def retrieve(question, k=TOP_K):
    qvec = embedder.encode([question], normalize_embeddings=True).tolist()
    results = collection.query(query_embeddings=qvec, n_results=min(k, collection.count()),
                               include=['documents', 'metadatas', 'distances'])
    return [{'text': txt, 'source': meta['source'], 'score': round(1 - float(dist), 3)}
            for txt, meta, dist in zip(results['documents'][0], results['metadatas'][0], results['distances'][0])]

sample_question = 'How quickly must a lost company laptop be reported?'
pd.DataFrame(retrieve(sample_question))

,text,source,score
0,Aster Robotics | Information Security Standard...,it_security.txt,0.372
1,Aster Robotics | Engineering Workflow | 2026 E...,engineering.txt,0.265
2,Aster Robotics | Human Resources Policy | 2026...,hr_policy.txt,0.252


## 8. Load pretrained LLM and implement RAG vs LLM-only

In [8]:
tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL)
llm = AutoModelForSeq2SeqLM.from_pretrained(GEN_MODEL).to(DEVICE)
llm.eval()

def generate(prompt, max_new_tokens=100):
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=480).to(DEVICE)
    with torch.inference_mode():
        output = llm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(output[0], skip_special_tokens=True).strip()

def answer_question(question, use_rag=True, k=TOP_K):
    evidence = retrieve(question, k) if use_rag else []
    if use_rag:
        context = '\n'.join(f"[{i+1}] ({e['source']}) {e['text']}" for i, e in enumerate(evidence))
        prompt = ("Answer the question using only the context. "
                  "If the answer is not in the context, say 'Not found in the documents'. "
                  "Give a short factual answer.\nContext:\n" + context + "\nQuestion: " + question + "\nAnswer:")
    else:
        prompt = 'Answer this question briefly from your own knowledge.\nQuestion: ' + question + '\nAnswer:'
    response = generate(prompt)
    return {'answer': response, 'sources': sorted(set(e['source'] for e in evidence)), 'evidence': evidence}

print('Question:', sample_question)
print('RAG:', answer_question(sample_question, True)['answer'])
print('LLM only:', answer_question(sample_question, False)['answer'])

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Question: How quickly must a lost company laptop be reported?
RAG: 30 minutes
LLM only: a few minutes


## 9. Side-by-side evaluation on held-out questions

In [9]:
TEST_CASES = [
 {'question':'How many paid annual leave days do employees receive?', 'expected':'18 days', 'keywords':['18'], 'source':'hr_policy.txt'},
 {'question':'Within how many minutes must a lost laptop be reported?', 'expected':'30 minutes', 'keywords':['30'], 'source':'it_security.txt'},
 {'question':'How many independent code reviewers are required for a release?', 'expected':'two reviewers', 'keywords':['two','2'], 'source':'engineering.txt'},
 {'question':'Where is the emergency assembly location?', 'expected':'east parking courtyard beside Gate C', 'keywords':['gate c','east parking'], 'source':'facilities.txt'},
 {'question':'What is the yearly employee training allowance?', 'expected':'1200 USD', 'keywords':['1200','1,200'], 'source':'learning.txt'},
 {'question':'How long is the engineering sprint cycle?', 'expected':'21 days', 'keywords':['21'], 'source':'engineering.txt'},
]

def keyword_match(answer, keywords):
    return int(any(k.lower() in answer.lower() for k in keywords))

rows=[]
for test in TEST_CASES:
    rag = answer_question(test['question'], True)
    base = answer_question(test['question'], False)
    rows.append({
        'Question': test['question'], 'Expected': test['expected'],
        'RAG response': rag['answer'], 'LLM-only response': base['answer'],
        'RAG correct (heuristic)': keyword_match(rag['answer'],test['keywords']),
        'LLM correct (heuristic)': keyword_match(base['answer'],test['keywords']),
        'Source retrieved': int(test['source'] in rag['sources']),
        'Cited sources': ', '.join(rag['sources'])
    })
results_df = pd.DataFrame(rows)
display(results_df)
print('RAG keyword accuracy:', round(results_df['RAG correct (heuristic)'].mean(),3))
print('LLM-only keyword accuracy:', round(results_df['LLM correct (heuristic)'].mean(),3))
print('Retrieval source hit rate:', round(results_df['Source retrieved'].mean(),3))

,Question,Expected,RAG response,LLM-only response,RAG correct (heuristic),LLM correct (heuristic),Source retrieved,Cited sources
0,How many paid annual leave days do employees r...,18 days,18 days,365,1,0,1,"hr_policy.txt, it_security.txt, learning.txt"
1,Within how many minutes must a lost laptop be ...,30 minutes,30,ten minutes,1,0,1,"engineering.txt, hr_policy.txt, it_security.txt"
2,How many independent code reviewers are requir...,two reviewers,two,a minimum of ten,1,0,1,"engineering.txt, it_security.txt, learning.txt"
3,Where is the emergency assembly location?,east parking courtyard beside Gate C,east parking courtyard beside Gate C,san francisco,1,0,1,"engineering.txt, facilities.txt, it_security.txt"
4,What is the yearly employee training allowance?,1200 USD,1200 USD,"£15,000",1,0,1,"hr_policy.txt, it_security.txt, learning.txt"
5,How long is the engineering sprint cycle?,21 days,21-day sprint cycle,a few seconds,1,0,1,"engineering.txt, hr_policy.txt, learning.txt"


RAG keyword accuracy: 1.0
LLM-only keyword accuracy: 0.0
Retrieval source hit rate: 1.0


### Evaluation interpretation and limitations
- **Keyword accuracy** checks whether an expected fact appears in the response; it is a lightweight proxy, not a full semantic correctness metric. Review the displayed answers manually.
- **Source hit rate** measures whether retrieval included the expected document, not whether the generated claim is fully supported.
- **Groundedness:** compare each RAG claim against its retrieved passages. Even RAG can hallucinate or misinterpret context.
- **LLM-only:** the model has no access to the fictional company's private policies, so confident specific claims may be unsupported.
- **Limitations:** tiny sample corpus, no reranker, character-based splitting, fixed top-k, small pretrained generator, and no formal human annotation. Scores vary with model/library versions.

## 10. Interactive application demonstration

In [10]:
question = input('Ask a question about Aster Robotics policies: ').strip() or 'What is the minimum company password length?'
rag = answer_question(question, True)
plain = answer_question(question, False)
print('\nQUESTION:', question)
print('\nRAG ANSWER:', rag['answer'])
print('SOURCES:', ', '.join(rag['sources']) or 'None')
print('\nLLM-ONLY ANSWER:', plain['answer'])
print('\nRETRIEVED EVIDENCE:')
for i, e in enumerate(rag['evidence'],1):
    print(f"{i}. {e['source']} | cosine similarity≈{e['score']:.3f}\n   {e['text']}\n")

Ask a question about Aster Robotics policies: How many paid annual leave days do employees receive?

QUESTION: How many paid annual leave days do employees receive?

RAG ANSWER: 18 days
SOURCES: hr_policy.txt, it_security.txt, learning.txt

LLM-ONLY ANSWER: 365

RETRIEVED EVIDENCE:
1. hr_policy.txt | cosine similarity≈0.559
   Aster Robotics | Human Resources Policy | 2026 Employees receive 18 days of paid annual leave each calendar year. Leave requests of three or more consecutive working days must be submitted at least 10 calendar days before the start date. The standard probation period for new employees is 90 days. Employees may work remotely up to two days per week with manager approval. All new hires must complete workplace safety orientation within their first 14 days.

2. learning.txt | cosine similarity≈0.250
   Aster Robotics | Learning and Development | 2026 Each employee receives a yearly training allowance of 1200 USD. Courses costing more than 300 USD require approval fro

In [11]:
question = input('Ask a question about Aster Robotics policies: ').strip() or 'What is the minimum company password length?'
rag = answer_question(question, True)
plain = answer_question(question, False)
print('\nQUESTION:', question)
print('\nRAG ANSWER:', rag['answer'])
print('SOURCES:', ', '.join(rag['sources']) or 'None')
print('\nLLM-ONLY ANSWER:', plain['answer'])
print('\nRETRIEVED EVIDENCE:')
for i, e in enumerate(rag['evidence'],1):
    print(f"{i}. {e['source']} | cosine similarity≈{e['score']:.3f}\n   {e['text']}\n")

Ask a question about Aster Robotics policies: Where is the emergency assembly location?

QUESTION: Where is the emergency assembly location?

RAG ANSWER: east parking courtyard beside Gate C
SOURCES: engineering.txt, facilities.txt, it_security.txt

LLM-ONLY ANSWER: san francisco

RETRIEVED EVIDENCE:
1. facilities.txt | cosine similarity≈0.369
   Aster Robotics | Facilities and Equipment | 2026 Prototype laboratory access is available from 07:00 to 21:00 on weekdays. Visitors must be escorted by an employee at all times. Protective eyewear is mandatory in the fabrication zone. Equipment incidents must be recorded in the SafetyLog portal within four hours. The emergency assembly location is the east parking courtyard beside Gate C.

2. engineering.txt | cosine similarity≈0.207
   Aster Robotics | Engineering Workflow | 2026 Every production robot software release requires two independent code reviewers and a passing automated safety test suite. The company uses a 21-day sprint cycle. Cr